# SiPM training

Trains the segmentation model — 3 classes: background, real damage, false artifact.
Shows the loss curve, some predicted overlays, and the damaged-vs-not table.

Two modes, set `MODE` down in the split cell:
- `holdout` — train on all trays but one, test on that unseen tray. This is the honest
  number. Got recall 0.94–0.96 at threshold 300.
- `final` — train on everything, no holdout. Only worth doing after holdout proves it
  generalizes. This is the model inference actually uses.

Drive needs the label JSONs, `cropped/`, and `sipm_dataset.py`.
Need the GPU for this one: Runtime → Change runtime type → GPU.

colab doesn't ship these two

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

mount drive + point at the folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/sipm'   # <-- change if needed
assert os.path.isdir(ROOT), f"Can't find {ROOT}"
import sys; sys.path.append(ROOT)

check there's actually a GPU. if it says cpu, go set the runtime and rerun

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)
if device=='cuda': print(torch.cuda.get_device_name(0))

load labels and split.

holdout keeps every chip from one tray out of training. Chips from the same tray look
alike so splitting randomly by chip leaks and makes the score look better than it is.

In [ ]:
import glob, random
from sipm_dataset import load_records, SiPMDataset

MODE     = 'holdout'          # 'holdout' (honest eval) or 'final' (train on everything)
VAL_TRAY = '250812-1302'      # tray held out when MODE='holdout'; rotate to cross-validate

records = load_records(sorted(glob.glob(os.path.join(ROOT,'*.json'))))
def tray_of(r): return r['key'].split('/')[0]      # key = "<tray>/<chip>.png"

print("trays found:", sorted(set(tray_of(r) for r in records)))

if MODE == 'holdout':
    train_recs = [r for r in records if tray_of(r) != VAL_TRAY]
    val_recs   = [r for r in records if tray_of(r) == VAL_TRAY]
    assert val_recs, f"No chips for {VAL_TRAY} - check the tray name."
else:                                   # final: no holdout
    train_recs = records
    val_recs   = records[:40]           # placeholder so the val cells run - IGNORE its numbers
    print("MODE='final': val numbers are meaningless; evaluation was done in holdout mode.")

random.seed(0); random.shuffle(train_recs)
dtr = sum(1 for r in train_recs if r['status']=='damaged')
dva = sum(1 for r in val_recs   if r['status']=='damaged')
print(f"train: {len(train_recs)} chips ({dtr} damaged)   val: {len(val_recs)} chips ({dva} damaged)")

datasets + augmentation.

only geometry stuff — flips, small rotate/shift/scale. Deliberately not doing heavy
brightness/contrast because that mimics glare and blurs the line between real damage
and artifacts, which is the whole thing the model has to tell apart.

In [ ]:
import albumentations as A
from torch.utils.data import DataLoader

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.95,1.05), translate_percent=0.03, rotate=(-12,12), p=0.7),
])

SIZE = 512
train_ds = SiPMDataset(train_recs, os.path.join(ROOT,'cropped'), size=SIZE, augment=train_aug)
val_ds   = SiPMDataset(val_recs,   os.path.join(ROOT,'cropped'), size=SIZE, augment=None)

train_dl = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2)
print("batches per epoch:", len(train_dl))

model + loss.

U-Net with a pretrained resnet34 encoder. Dice + focal because damage is a tiny
fraction of the pixels and plain cross-entropy would just predict background everywhere.

In [ ]:
import segmentation_models_pytorch as smp
model = smp.Unet('resnet34', encoder_weights='imagenet', classes=3, in_channels=3).to(device)
dice  = smp.losses.DiceLoss(mode='multiclass')
focal = smp.losses.FocalLoss(mode='multiclass')
def loss_fn(logits, y): return dice(logits, y) + focal(logits, y)
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler('cuda', enabled=(device=='cuda'))

train. 15 epochs is a few minutes on the colab GPU. watch the loss drop.

In [ ]:
EPOCHS = 15
hist = []
for ep in range(EPOCHS):
    model.train(); tot=0
    for x,y,_ in train_dl:
        x,y = x.to(device), y.to(device)
        opt.zero_grad()
        with torch.autocast(device_type='cuda', enabled=(device=='cuda')):
            loss = loss_fn(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item()
    # quick val loss
    model.eval(); vtot=0
    with torch.no_grad():
        for x,y,_ in val_dl:
            x,y=x.to(device),y.to(device)
            vtot += loss_fn(model(x), y).item()
    tr, va = tot/len(train_dl), vtot/max(1,len(val_dl))
    hist.append((tr,va))
    print(f"epoch {ep+1:2d}/{EPOCHS}  train {tr:.4f}   val {va:.4f}")

loss curve

In [ ]:
import matplotlib.pyplot as plt
tr=[h[0] for h in hist]; va=[h[1] for h in hist]
plt.plot(tr,label='train'); plt.plot(va,label='val')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title('training'); plt.show()

look at what it predicts on chips it never trained on.
red = predicted damage, blue = predicted artifact.

In [ ]:
import numpy as np
from PIL import Image
from sipm_dataset import SiPMDataset as DS

model.eval()
show = val_recs[:6]
fig,axes=plt.subplots(2,3,figsize=(13,9))
for ax,r in zip(axes.flat, show):
    x,y,_ = val_ds[val_recs.index(r)]
    with torch.no_grad():
        pred = model(x.unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()
    img = DS._pad_square(Image.open(os.path.join(ROOT,'cropped',r['key'])).convert('RGB'),0).resize((SIZE,SIZE))
    ov=np.array(img).astype(float)
    ov[pred==1]=0.5*ov[pred==1]+0.5*np.array([255,0,0])
    ov[pred==2]=0.5*ov[pred==2]+0.5*np.array([0,0,255])
    ax.imshow(ov.astype('uint8')); ax.set_title(r['key']+f"  (true grade {r['grade']})",fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

turn the pixel map into a yes/no by counting predicted damage pixels, and sweep the
cutoff. Recall = how much of the real damage it caught, which is the one I care about
most — missing a bad chip is worse than a false alarm someone glances at.

In [ ]:
import numpy as np
model.eval()
dmg_px, truth = [], []
with torch.no_grad():
    for i,r in enumerate(val_recs):
        x,_,_ = val_ds[i]
        pred = model(x.unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()
        dmg_px.append(int((pred==1).sum()))
        truth.append(1 if r['status']=='damaged' else 0)
dmg_px=np.array(dmg_px); truth=np.array(truth)

print("threshold | recall(catch dmg) | precision | false-alarms")
for thr in [10,50,100,300,800]:
    pred_dmg = (dmg_px>thr).astype(int)
    tp=int(((pred_dmg==1)&(truth==1)).sum()); fp=int(((pred_dmg==1)&(truth==0)).sum())
    fn=int(((pred_dmg==0)&(truth==1)).sum())
    rec = tp/(tp+fn) if tp+fn else 0; prec = tp/(tp+fp) if tp+fp else 0
    print(f"  {thr:5d}   |   {rec:4.2f}            |  {prec:4.2f}     |  {fp}")
print("\nPick the threshold that catches most damage with few false alarms (bias to recall).")

save it. filename gets the date + which tray was held out so a new run doesn't
overwrite a good one. Write down the threshold and recall from the table above.

In [ ]:
import time
tag  = time.strftime('%Y%m%d_%H%M')
name = (f'sipm_unet_{tag}_final_alltrays.pt' if MODE=='final'
        else f'sipm_unet_{tag}_val-{VAL_TRAY}.pt')
torch.save(model.state_dict(), os.path.join(ROOT, name))
print('saved:', name)

---
If the loss dropped and the overlays are landing on real damage, the whole thing works.

holdout mode → the table above is the honest score on an unseen tray. Rotate `VAL_TRAY`
through each tray to cross-validate.
final mode → this saved model is what run_inference.py loads.

The threshold isn't part of the model, it gets applied after, so any saved model can be
re-thresholded later by rerunning that cell.

Still thin on grade 4–5 (~29 chips) so it's weakest on the really severe stuff.